# Чистый ноутбук для fine-tuning LEGATO

Этот ноутбук рассчитан на уже подготовленное совместимое окружение и не меняет файлы проекта.

Что делает:
- читает датасет с колонками `image_path` и `abc_path` или уже готовыми `image`/`transcription`
- анализирует словарь ABC
- добавляет маркеры `<TITLE>`, `<LYRICS>`, `<ANNOT>`
- загружает чекпоинт LEGATO
- замораживает почти всю модель для теста
- обучает с взвешенным loss


In [16]:
from __future__ import annotations

import json
import os
import string
import sys
import unicodedata
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
import re

import numpy as np
import torch
import torch.distributed as dist
from datasets import Dataset, DatasetDict, load_from_disk
from huggingface_hub import login, snapshot_download
from PIL import Image
from safetensors.torch import load_file as load_safetensors_file
from torch import nn
from transformers import AutoConfig, AutoModel, AutoProcessor, Seq2SeqTrainingArguments, TrainerCallback, set_seed
from transformers import MllamaVisionModel


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'legato').exists() and (candidate / 'scripts' / 'train.py').exists():
            return candidate
    raise FileNotFoundError('Не удалось найти корень проекта LEGATO.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Важно: импортируем legato.models до AutoProcessor/AutoModel.from_pretrained,
# чтобы сработала регистрация кастомных классов.
from legato.metrics import compute_error_rates
from legato.models import LegatoModel, LegatoConfig
from legato.trainer import LegatoTrainer

HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('HF token загружен из окружения.')
else:
    print('HF_TOKEN не найден. Если вы уже залогинены через huggingface_hub, это не страшно.')

NOTEBOOK_ROOT = PROJECT_ROOT / 'legato-utils-new'
ARTIFACT_ROOT = NOTEBOOK_ROOT / 'artifacts'
OUTPUT_ROOT = NOTEBOOK_ROOT / 'outputs'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('NOTEBOOK_ROOT =', NOTEBOOK_ROOT)


def load_repo_state_dict(repo_id: str, token: str | None = None) -> dict[str, torch.Tensor]:
    snapshot_path = Path(snapshot_download(repo_id=repo_id, token=token))
    safetensors_files = sorted(snapshot_path.glob('*.safetensors'))
    if not safetensors_files:
        raise FileNotFoundError(f'В snapshot не найдено *.safetensors: {snapshot_path}')
    merged = {}
    for safetensors_path in safetensors_files:
        merged.update(load_safetensors_file(str(safetensors_path)))
    return merged


def remap_legato_keys_for_current_mllama(state_dict: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    remapped = {}
    for key, value in state_dict.items():
        new_key = key
        if key.startswith('language_model.model.'):
            new_key = 'model.language_model.' + key[len('language_model.model.'):]
        elif key.startswith('language_model.lm_head.'):
            new_key = 'lm_head.' + key[len('language_model.lm_head.'):]
        elif key.startswith('language_model.'):
            new_key = 'model.' + key
        elif key.startswith('multi_modal_projector.'):
            new_key = 'model.multi_modal_projector.' + key[len('multi_modal_projector.'):]
        elif key.startswith('vision_model.'):
            # Не грузим vision из LEGATO-чекпоинта: ниже загрузим encoder отдельно из base vision model.
            continue
        remapped[new_key] = value
    return remapped


def load_legato_model_clean(model_ref: str, token: str | None = None):
    config = AutoConfig.from_pretrained(model_ref, token=token)
    assert isinstance(config, LegatoConfig), f'Ожидался LegatoConfig, получили: {type(config)}'
    model = LegatoModel(config, load_pretrained_encoder=False)

    raw_state_dict = load_repo_state_dict(model_ref, token=token)
    remapped_state_dict = remap_legato_keys_for_current_mllama(raw_state_dict)
    incompatible = model.load_state_dict(remapped_state_dict, strict=False)

    encoder_ref = getattr(config, 'encoder_pretrained_model_name_or_path', None)
    if encoder_ref is None:
        raise ValueError('В конфиге отсутствует encoder_pretrained_model_name_or_path')
    model.model.vision_model = MllamaVisionModel.from_pretrained(encoder_ref, token=token)
    model.config.vision_config = model.model.vision_model.config
    for param in model.model.vision_model.parameters():
        param.requires_grad = False

    print('missing keys:', len(incompatible.missing_keys))
    print('unexpected keys:', len(incompatible.unexpected_keys))
    if incompatible.missing_keys:
        print('Первые missing keys:', incompatible.missing_keys[:20])
    if incompatible.unexpected_keys:
        print('Первые unexpected keys:', incompatible.unexpected_keys[:20])

    return model


HF_TOKEN не найден. Если вы уже залогинены через huggingface_hub, это не страшно.
PROJECT_ROOT = F:\legato
NOTEBOOK_ROOT = F:\legato\legato-utils-new


In [17]:
@dataclass
class PipelineConfig:
    base_model: str = 'guangyangmusic/legato'
    dataset_path: str = r'F:\datasets\PDMX\dataset_root\hf_dataset'
    output_name: str = 'legato-text-music-clean-tiny'
    seed: int = 42
    abc_encoding: str = 'utf-8'
    markers: tuple[str, ...] = tuple()
    force_add_characters: tuple[str, ...] = ('&', ';', '@', 'J', 'W', 'Y', 'Z', 'j', '\xa0', '´')
    music_loss_weight: float = 0.75
    train_split_name: str = 'train'
    val_split_name: str = 'validation'
    test_split_name: str = 'test'
    max_train_samples: int = 7664
    max_val_samples: int = 64
    max_test_samples: int = 8
    num_train_epochs: int = 1
    max_steps: int = 900
    learning_rate: float = 3e-5
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 4
    dataloader_num_workers: int = 0
    generation_max_length: int = 512
    generation_num_beams: int = 1
    logging_steps: int = 25
    eval_steps: int = 100
    save_steps: int = 200
    train_last_n_decoder_layers: int = 2
    train_lm_head: bool = True
    train_token_embeddings: bool = True
    run_train: bool = True
    run_eval: bool = True


CFG = PipelineConfig()
set_seed(CFG.seed)

DATASET_PATH = CFG.dataset_path
RUN_ROOT = OUTPUT_ROOT / CFG.output_name
PROCESSOR_OUT = RUN_ROOT / 'processor_extended'
MODEL_OUT = RUN_ROOT / 'model'
METRICS_LOG_PATH = RUN_ROOT / 'metrics_log.txt'
REPORT_PATH = ARTIFACT_ROOT / f'{CFG.output_name}_vocab_report.json'
META_PATH = ARTIFACT_ROOT / f'{CFG.output_name}_tokenizer_meta.json'

RUN_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSOR_OUT.mkdir(parents=True, exist_ok=True)
MODEL_OUT.mkdir(parents=True, exist_ok=True)
METRICS_LOG_PATH.write_text('', encoding='utf-8')

print(CFG)


PipelineConfig(base_model='guangyangmusic/legato', dataset_path='F:\\datasets\\PDMX\\dataset_root\\hf_dataset', output_name='legato-text-music-clean-tiny', seed=42, abc_encoding='utf-8', markers=(), force_add_characters=('&', ';', '@', 'J', 'W', 'Y', 'Z', 'j', '\xa0', '´'), music_loss_weight=0.75, train_split_name='train', val_split_name='validation', test_split_name='test', max_train_samples=7664, max_val_samples=64, max_test_samples=8, num_train_epochs=1, max_steps=900, learning_rate=3e-05, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=4, dataloader_num_workers=0, generation_max_length=512, generation_num_beams=1, logging_steps=25, eval_steps=100, save_steps=200, train_last_n_decoder_layers=2, train_lm_head=True, train_token_embeddings=True, run_train=True, run_eval=True)


## 1. Нормализация датасета

In [18]:
def resolve_existing_path(raw_path: str | Path, base_dir: Path) -> Path:
    candidate = Path(raw_path)
    if candidate.is_absolute() and candidate.exists():
        return candidate
    variants = [base_dir / candidate, PROJECT_ROOT / candidate, candidate]
    for path in variants:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f'Не найден путь: {raw_path}')


NAME_ATTR_RE = re.compile(r'\b(?:nm|snm)="([^"]+)"')


def build_transcription_from_abc(abc_text: str) -> str:
    abc_text = abc_text.replace('\r', '\n').replace('\xa0', ' ')
    cleaned_lines = []

    for raw_line in abc_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        # Убираем lyrics-строки, чтобы модель не тонула в длинном текстовом блоке.
        if line.startswith('w:'):
            continue

        # Оставляем нативный ABC-формат, но убираем подписи инструментов из V-строк.
        if line.startswith('V:'):
            line = re.sub(r'\s+nm="[^"]*"', '', line)
            line = re.sub(r'\s+snm="[^"]*"', '', line)
            line = re.sub(r'\s+', ' ', line).strip()

        cleaned_lines.append(line)

    return '\n'.join(cleaned_lines)


def load_transcription_from_split(split_ds):
    if 'transcription' in split_ds.column_names:
        return [str(x) if x is not None else '' for x in split_ds['transcription']]

    if 'abc' in split_ds.column_names:
        return [build_transcription_from_abc(str(x)) if x is not None else '' for x in split_ds['abc']]

    for key in ['abc_path', 'transcription_path', 'label_path', 'abc_file', 'annotation_path']:
        if key in split_ds.column_names:
            paths = split_ds[key]
            outputs = []
            for raw_path in paths:
                if raw_path is None:
                    outputs.append('')
                    continue
                abc_file = resolve_existing_path(raw_path, DATASET_PATH)
                outputs.append(build_transcription_from_abc(abc_file.read_text(encoding=CFG.abc_encoding)))
            return outputs

    raise KeyError(f"?? ??????? ???? ? ABC/?????????????. ????????? ????: {split_ds.column_names}")


def normalize_split(split_ds, split_name: str):
    transcriptions = load_transcription_from_split(split_ds)
    normalized = split_ds
    if 'transcription' in normalized.column_names:
        normalized = normalized.remove_columns(['transcription'])
    if 'split_name' in normalized.column_names:
        normalized = normalized.remove_columns(['split_name'])
    normalized = normalized.add_column('transcription', transcriptions)
    normalized = normalized.add_column('split_name', [split_name] * len(normalized))
    return normalized


raw_dataset = load_from_disk(str(DATASET_PATH))
assert isinstance(raw_dataset, DatasetDict), 'Ожидался DatasetDict с train/validation/test.'

print('Найденные сплиты:', list(raw_dataset.keys()))

train_raw = raw_dataset[CFG.train_split_name].select(range(min(CFG.max_train_samples, len(raw_dataset[CFG.train_split_name]))))
val_raw = raw_dataset[CFG.val_split_name].select(range(min(CFG.max_val_samples, len(raw_dataset[CFG.val_split_name]))))
test_raw = raw_dataset[CFG.test_split_name].select(range(min(CFG.max_test_samples, len(raw_dataset[CFG.test_split_name]))))

dataset = DatasetDict({
    'train': normalize_split(train_raw, 'train'),
    'val': normalize_split(val_raw, 'val'),
    'test': normalize_split(test_raw, 'test'),
})

for split_name, split_ds in dataset.items():
    print(split_name, len(split_ds), split_ds.column_names)


Найденные сплиты: ['train', 'validation', 'test']
train 7664 ['id', 'image', 'abc', 'transcription', 'split_name']
val 64 ['id', 'image', 'abc', 'transcription', 'split_name']
test 8 ['id', 'image', 'abc', 'transcription', 'split_name']


In [5]:
print(dataset['train'][0]['transcription'][:4000])
print(dataset['val'][1]['transcription'][:4000])

X:1
T:James Gibb editions
T:Venite amanti - Festa
%%score [ 1 2 3 ]
L:1/8
Q:1/4=120
M:2/2
I:linebreak $
K:F
V:1 treble
V:2 treble
V:3 treble
V:1
"^Venite amanti""^Costanzo Festa\n(1490-1545)""^Angelo Poliziano\n(1454-97)" z8 | z4 D4 | E2 D2 G4 | F4 (G2 B2- | B2 AG) A4 | %5
z2 (G4 FE) | (D3 E F2) G2 |$[M:2/2] (B6 AG | A4) G2 c2 | B2 G2 A4 | %10
G4 z2 G2 | B4 c4 | d4 B2 c2- | c2 A2 B4 |$ G2 A4 F2 | %15
B4 G2 (c2- | c2 A2 B4 | A3 B c2 d2- | d2 ^c2) d2 G2 | A2 B2 (c3 B |$ %20
AG) (F2 G2 A2 |$ D2 F2 E4) | (D3 E FG (A2 | AB) c2 B2 AG) | F2 G2 B4 | %25
A2 d4 B2 | (c3 B/A/) (G3 A |$ Bc de f2) (e2- | ed) (d4 ^c2) | d2 d2 z2 (D2- | %30
DEFG A2) B2 | A2 (c3 B AG | F3 G A2) B2 |$ c2 d2 B2 c2 | (d3 c BA G2 | %35
B2 c2 A4) | G8 || d8- | d4 e4 | f4 d4 |$ %40
B3 c d2 c2- | c2 B2 A4 | G8 | z4 B4 | A4 G4 | %45
F4 B4 | A3 B c2 d2- |$ d2 ^c2 d4 | G4 A4 | B4 G2 d2 | %50
d2 c2 _e2 d2 | B4 c2 c2 | B2 c2 d2 B2 |$ A4 G2 c2- | c2 B2 A4 | %55
G2 G2 d2 e2 |[Q:1/4=118] f3[Q:1/4=115] e[Q:1/4=114] d2[Q:1/4=111] c2 |

## 2. Анализ словаря и расширение tokenizer

In [19]:
base_processor = AutoProcessor.from_pretrained(CFG.base_model, token=HF_TOKEN)
base_tokenizer = base_processor.tokenizer


def iter_transcriptions(ds_dict: DatasetDict) -> Iterable[str]:
    for split_ds in ds_dict.values():
        for item in split_ds['transcription']:
            if item is not None:
                yield item


def contains_unknown_token(tokenizer, text: str) -> bool:
    ids = tokenizer(text, add_special_tokens=False)['input_ids']
    if not ids:
        return True
    tokens = tokenizer.convert_ids_to_tokens(ids)
    unk_token = getattr(tokenizer, 'unk_token', None)
    if unk_token and unk_token in tokens:
        return True
    normalized_tokens = [str(tok).lower() for tok in tokens]
    return any('unk' in tok for tok in normalized_tokens)


char_counter = Counter()
num_transcriptions_seen = 0
for item in iter_transcriptions(dataset):
    char_counter.update(item)
    num_transcriptions_seen += 1
unique_chars = sorted(char_counter)
forced_chars = sorted(set(CFG.force_add_characters))

dataset_chars_with_unk = sorted(ch for ch in unique_chars if contains_unknown_token(base_tokenizer, ch))
forced_chars_with_unk = sorted(ch for ch in forced_chars if contains_unknown_token(base_tokenizer, ch))
all_chars_with_unk = sorted(set(dataset_chars_with_unk) | set(forced_chars_with_unk))
chars_to_add = sorted(ch for ch in forced_chars_with_unk)
excluded_chars_with_unk = sorted(ch for ch in all_chars_with_unk if ch not in chars_to_add)

report = {
    'base_model': CFG.base_model,
    'dataset_path': str(DATASET_PATH),
    'num_train_items': len(dataset['train']),
    'num_val_items': len(dataset['val']),
    'num_test_items': len(dataset['test']),
    'num_transcriptions_seen': num_transcriptions_seen,
    'markers': list(CFG.markers),
    'chars_to_add': chars_to_add,
    'dataset_chars_with_unk': dataset_chars_with_unk,
    'forced_chars_with_unk': forced_chars_with_unk,
    'excluded_chars_with_unk': excluded_chars_with_unk,
    'top_100_char_counts': dict(char_counter.most_common(100)),
}
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')

processor = AutoProcessor.from_pretrained(CFG.base_model, token=HF_TOKEN)
tokenizer = processor.tokenizer
original_vocab_size = len(tokenizer)

existing_vocab = tokenizer.get_vocab()
marker_tokens_to_add = [tok for tok in CFG.markers if tok not in existing_vocab]
char_tokens_to_add = [tok for tok in chars_to_add if tok not in existing_vocab]

num_added_markers = tokenizer.add_special_tokens({'additional_special_tokens': marker_tokens_to_add}) if marker_tokens_to_add else 0
num_added_chars = tokenizer.add_tokens(char_tokens_to_add) if char_tokens_to_add else 0
processor.save_pretrained(PROCESSOR_OUT)

tokenizer_meta = {
    'base_model': CFG.base_model,
    'processor_out': str(PROCESSOR_OUT),
    'original_vocab_size': original_vocab_size,
    'extended_vocab_size': len(tokenizer),
    'num_added_markers': num_added_markers,
    'num_added_chars': num_added_chars,
    'marker_tokens': list(CFG.markers),
    'marker_token_ids': [tokenizer.convert_tokens_to_ids(tok) for tok in CFG.markers],
    'dataset_chars_with_unk': dataset_chars_with_unk,
    'forced_chars_with_unk': forced_chars_with_unk,
    'excluded_chars_with_unk': excluded_chars_with_unk,
    'char_tokens_added': char_tokens_to_add,
}
META_PATH.write_text(json.dumps(tokenizer_meta, indent=2, ensure_ascii=False), encoding='utf-8')
print('unk_token:', getattr(tokenizer, 'unk_token', None))
print('unk_token_id:', getattr(tokenizer, 'unk_token_id', None))
print('Символы из датасета с unk:', dataset_chars_with_unk)
print('Принудительно проверенные символы с unk:', forced_chars_with_unk)
print('Исключенные символы с unk:', excluded_chars_with_unk)
print('Символы, выбранные для добавления:', char_tokens_to_add)
print(json.dumps(tokenizer_meta, indent=2, ensure_ascii=False))


unk_token: None
unk_token_id: None
Символы из датасета с unk: ['&', ';', '@', 'J', 'W', 'Y', 'Z', '`', 'j', '¡', '£', '§', '©', 'ª', '«', '°', '³', '´', '·', 'º', '»', '¼', '¿', 'À', 'Á', 'Ã', 'Ä', 'Å', 'Ç', 'É', 'Í', 'Î', 'Ó', 'Ö', 'Ø', 'Ú', 'Ü', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ò', 'ó', 'ô', 'õ', 'ö', 'ø', 'ù', 'ú', 'û', 'ü', 'ý', 'þ', 'ā', 'Ă', 'ă', 'ą', 'ć', 'Č', 'č', 'ď', 'Đ', 'đ', 'ē', 'ę', 'ě', 'Ğ', 'ģ', 'Ī', 'ī', 'İ', 'ı', 'ļ', 'ľ', 'Ł', 'ł', 'ń', 'ņ', 'ň', 'ő', 'Ř', 'ř', 'ś', 'Ş', 'Š', 'š', 'ť', 'Ů', 'ů', 'ű', 'Ż', 'ż', 'ž', 'ſ', 'ơ', 'ư', 'Ș', 'ș', 'ț', '̀', '́', '̃', 'Α', 'Β', 'Γ', 'Ε', 'Ζ', 'Η', 'Θ', 'Ι', 'Κ', 'Λ', 'Μ', 'Ν', 'Ο', 'Ρ', 'Σ', 'Τ', 'Υ', 'γ', 'ο', 'π', 'І', 'А', 'Б', 'В', 'Г', 'Д', 'Е', 'Ж', 'З', 'И', 'Й', 'К', 'Л', 'М', 'Н', 'О', 'П', 'Р', 'С', 'Т', 'У', 'Ф', 'Х', 'Ц', 'Ш', 'Щ', 'Ь', 'Э', 'Ю', 'Я', 'а', 'б', 'в', 'г', 'д', 'е', 'ж', 'з', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 'ф

## 3. Взвешенный trainer

In [20]:
def build_token_weight_vector(vocab_size: int, downweighted_token_ids: list[int], default_weight: float, downweight: float) -> torch.Tensor:
    weights = torch.full((vocab_size,), float(default_weight), dtype=torch.float32)
    if downweighted_token_ids:
        weights[torch.tensor(downweighted_token_ids, dtype=torch.long)] = float(downweight)
    return weights


special_ids = set(tokenizer.all_special_ids)
downweighted_token_ids = [idx for idx in range(original_vocab_size) if idx not in special_ids]
token_weight_vector = build_token_weight_vector(
    vocab_size=len(tokenizer),
    downweighted_token_ids=downweighted_token_ids,
    default_weight=1.0,
    downweight=CFG.music_loss_weight,
)


class WeightedLegatoTrainer(LegatoTrainer):
    def __init__(self, *args, token_weight_vector: torch.Tensor, **kwargs):
        super().__init__(*args, **kwargs)
        self._token_weight_vector = token_weight_vector.clone().float()
        self._token_weight_vector_device = None

    def _get_token_weight_vector(self, device: torch.device) -> torch.Tensor:
        if self._token_weight_vector_device != device:
            self._token_weight_vector = self._token_weight_vector.to(device)
            self._token_weight_vector_device = device
        return self._token_weight_vector

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = {k: v for k, v in inputs.items() if not k.startswith('gen_')}
        labels = inputs['labels']
        model_inputs = {k: v for k, v in inputs.items() if k != 'labels'}
        outputs = model(**model_inputs, use_cache=False)
        logits = outputs.logits.float()

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = nn.CrossEntropyLoss(reduction='none')
        token_losses = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        ).view_as(shift_labels)

        weight_vector = self._get_token_weight_vector(shift_logits.device)
        safe_labels = shift_labels.clamp(min=0)
        token_weights = weight_vector[safe_labels]
        token_weights = torch.where(shift_labels.eq(-100), torch.zeros_like(token_weights), token_weights)

        weighted_loss = (token_losses * token_weights).sum() / token_weights.sum().clamp_min(1e-8)
        return (weighted_loss, outputs) if return_outputs else weighted_loss


class TextMetricsLoggerCallback(TrainerCallback):
    def __init__(self, log_path: Path):
        self.log_path = Path(log_path)

    def _append(self, payload: dict):
        with self.log_path.open('a', encoding='utf-8') as f:
            f.write(json.dumps(payload, ensure_ascii=False) + '\n')

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            payload = {'event': 'log', 'step': state.global_step, **logs}
            self._append(payload)

    def on_save(self, args, state, control, **kwargs):
        payload = {
            'event': 'save',
            'step': state.global_step,
            'checkpoint_dir': str(Path(args.output_dir) / f'checkpoint-{state.global_step}'),
        }
        self._append(payload)


## 4. Загрузка модели и сильная заморозка

In [21]:
model = load_legato_model_clean(CFG.base_model, token=HF_TOKEN)
model.resize_token_embeddings(len(tokenizer))


def freeze_all_parameters(model):
    for param in model.parameters():
        param.requires_grad = False


def get_decoder_layers(model):
    candidates = [
        ('model.language_model.layers', getattr(getattr(getattr(model, 'model', None), 'language_model', None), 'layers', None)),
        ('model.language_model.model.layers', getattr(getattr(getattr(getattr(model, 'model', None), 'language_model', None), 'model', None), 'layers', None)),
        ('language_model.layers', getattr(getattr(model, 'language_model', None), 'layers', None)),
        ('language_model.model.layers', getattr(getattr(getattr(model, 'language_model', None), 'model', None), 'layers', None)),
    ]
    for name, layers in candidates:
        if layers is not None:
            return name, layers
    raise AttributeError('Не удалось найти слои декодера.')


freeze_all_parameters(model)
for param in model.model.vision_model.parameters():
    param.requires_grad = False

decoder_path, decoder_layers = get_decoder_layers(model)
for layer in list(decoder_layers)[-CFG.train_last_n_decoder_layers:]:
    for param in layer.parameters():
        param.requires_grad = True

if CFG.train_token_embeddings:
    for param in model.get_input_embeddings().parameters():
        param.requires_grad = True

if CFG.train_lm_head:
    for param in model.get_output_embeddings().parameters():
        param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print('Путь к слоям декодера:', decoder_path)
print('Обучаемые параметры:', f'{trainable_params:,}')
print('Замороженные параметры:', f'{frozen_params:,}')


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


missing keys: 0
unexpected keys: 0
Путь к слоям декодера: model.language_model.layers
Обучаемые параметры: 16,882,176
Замороженные параметры: 926,778,907


## 5. Trainer и tiny-конфиг

In [22]:
def get_metric_target(examples):
    return {
        'label_ids': tokenizer(
            examples['transcription'],
            add_special_tokens=False,
            truncation=False,
        )['input_ids'],
    }


map_num_proc = CFG.dataloader_num_workers if CFG.dataloader_num_workers > 0 else None

metric_targets = dataset['val'].map(
    get_metric_target,
    remove_columns=dataset['val'].column_names,
    num_proc=map_num_proc,
    batched=True,
).to_dict()
tokens_to_mask = torch.tensor([tokenizer.convert_tokens_to_ids(tok) for tok in CFG.markers] + [tokenizer.pad_token_id])


def collate_fn(examples):
    outputs = processor(
        images=[example['image'].convert('RGB') if hasattr(example['image'], 'convert') else example['image'] for example in examples],
        text=[example['transcription'] for example in examples],
        return_num_tiles=True,
        truncation=True,
        padding='max_length',
        return_tensors='pt',
    )
    gen_outputs = processor(
        num_tiles=outputs.pop('num_tiles'),
        truncation=True,
        padding=True,
        return_tensors='pt',
    )
    outputs.update({
        f'gen_{k}': outputs[k] if k not in gen_outputs else gen_outputs[k]
        for k in outputs
    })
    outputs['labels'] = outputs['input_ids'].clone().masked_fill(torch.isin(outputs['input_ids'], tokens_to_mask), -100)
    return outputs


special_tokens = [tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id, -100]


def remove_special_tokens(array):
    masks = np.isin(array, special_tokens, invert=True)
    return [a[mask] for a, mask in zip(array, masks)]


def metric_fn(p):
    preds = remove_special_tokens(p.predictions)
    metric_num_workers = max(1, CFG.dataloader_num_workers)
    results = [
        compute_error_rates(
            tokenizer,
            metric_num_workers,
            *metric_targets.values(),
            preds,
        )
    ] if getattr(training_args, 'process_index', 0) == 0 else [None]

    if dist.is_available() and dist.is_initialized():
        dist.broadcast_object_list(results, src=0)

    return results[0]



training_args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_OUT),
    remove_unused_columns=False,
    do_train=True,
    do_eval=True,
    predict_with_generate=True,
    metric_for_best_model='eval_SER',
    greater_is_better=False,
    load_best_model_at_end=False,
    num_train_epochs=CFG.num_train_epochs,
    max_steps=CFG.max_steps,
    learning_rate=CFG.learning_rate,
    per_device_train_batch_size=CFG.per_device_train_batch_size,
    per_device_eval_batch_size=CFG.per_device_eval_batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    dataloader_num_workers=CFG.dataloader_num_workers,
    logging_steps=CFG.logging_steps,
    save_strategy='steps',
    save_total_limit=2,
    save_steps=CFG.save_steps,
    eval_strategy='steps',
    eval_steps=CFG.eval_steps,
    generation_max_length=CFG.generation_max_length,
    generation_num_beams=CFG.generation_num_beams,
    bf16=torch.cuda.is_available(),
    bf16_full_eval=torch.cuda.is_available(),
    report_to='none',
    seed=CFG.seed,
    logging_dir=str(MODEL_OUT / 'logs'),
)

trainer = WeightedLegatoTrainer(
    model=model,
    args=training_args,
    token_weight_vector=token_weight_vector,
    data_collator=collate_fn,
    train_dataset=dataset['train'],
    eval_dataset=dataset['val'],
    compute_metrics=metric_fn,
)

trainer.add_callback(TextMetricsLoggerCallback(METRICS_LOG_PATH))

print('Trainer готов.')
print('Metrics log:', METRICS_LOG_PATH)


Trainer готов.
Metrics log: F:\legato\legato-utils-new\outputs\legato-text-music-clean-tiny\metrics_log.txt


## 6. Запуск

In [9]:
RUN_TRAIN = CFG.run_train
RUN_EVAL = CFG.run_eval

if RUN_TRAIN:
    train_result = trainer.train()
    trainer.save_model(str(MODEL_OUT / 'final'))
    processor.save_pretrained(MODEL_OUT / 'final')
    with open(MODEL_OUT / 'train_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(train_result.metrics, f, indent=2)
    print('Обучение завершено.')

if RUN_EVAL:
    eval_metrics = trainer.evaluate()
    with open(MODEL_OUT / 'eval_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(eval_metrics, f, indent=2)
    print(eval_metrics)


Step,Training Loss,Validation Loss,Ler,Cer,Ser
100,1.509600,1.598070,77.691958,74.261501,76.112455
200,1.007900,1.121314,79.054975,76.040297,78.183192
300,1.040000,0.958678,87.687415,82.882221,89.149033
400,0.771600,0.886092,83.462063,79.860775,82.858222
500,0.859900,0.839099,78.191731,75.549118,79.383313
600,0.818300,0.808592,77.873694,76.074023,80.116385
700,0.825100,0.789042,78.100863,77.556209,81.313482
800,0.842900,0.777974,79.009541,77.992044,82.336759
900,0.751900,0.774004,78.282599,77.666033,81.958888


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
computing LER...: 100%|████████████████████████████████████████████████████████████████| 64/64 [00:02<00:00, 27.33it/s]


Обучение завершено.


computing LER...: 100%|████████████████████████████████████████████████████████████████| 64/64 [00:02<00:00, 27.01it/s]

{'eval_loss': 0.7739594578742981, 'eval_LER': 77.10131758291685, 'eval_CER': 77.6245243860256, 'eval_SER': 82.54232164449819, 'eval_runtime': 720.7517, 'eval_samples_per_second': 0.089, 'eval_steps_per_second': 0.089, 'epoch': 0.4697286012526096}


Большое количество warning при перезагрузке лучшего чекпоинта из-за того, что vision_model не сохраняется в checkpoint

это неприятно, но не означает, что сам train сломан

это можно игнорировать или просто выключить load_best_model_at_end

In [10]:
def preview_predictions(num_examples: int = 1):
    subset = dataset['val'].select(range(min(num_examples, len(dataset['val']))))
    outputs = trainer.predict(subset)

    preds = outputs.predictions
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)

    decoded = processor.batch_decode(preds, skip_special_tokens=False)

    for idx, sample in enumerate(decoded):
        print('=' * 80)
        print(f'Пример {idx}')
        print(sample[:4000])
        print('\nGT:')
        print(subset[idx]['transcription'][:4000])


preview_predictions(2)


computing LER...: 100%|██████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.20s/it]


Пример 0
<|begin_of_abc|>X:1
T:Title
%%score
L:1/8
Q:1/4=108
M:4/4
I:linebreak $
K:C
V:1 treble
V:1
 [GA]2 A2 B2 c2 B4 A2 A2 A2 B2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2

In [26]:
CKPT_DIR = Path(r"F:\legato\legato-utils-new\outputs\legato-text-music-clean-tiny\model\checkpoint-900")
PROC_DIR = Path(r"F:\legato\legato-utils-new\outputs\legato-text-music-clean-tiny\processor_extended")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Загружаем именно тот processor, с которым учился checkpoint
processor = AutoProcessor.from_pretrained(str(PROC_DIR))
tokenizer = processor if not hasattr(processor, "tokenizer") else processor.tokenizer

print("saved tokenizer size:", len(tokenizer))

# 2. Собираем базовую модель
model = load_legato_model_clean(CFG.base_model, token=HF_TOKEN)

# 3. Подгоняем embeddings под vocab checkpoint-а
model.resize_token_embeddings(len(tokenizer))

# 4. Загружаем trainer checkpoint
raw_ckpt_state = load_safetensors_file(str(CKPT_DIR / "model.safetensors"))
remapped_ckpt_state = remap_legato_keys_for_current_mllama(raw_ckpt_state)

incompatible = model.load_state_dict(remapped_ckpt_state, strict=False)

print("checkpoint missing keys:", len(incompatible.missing_keys))
print("checkpoint unexpected keys:", len(incompatible.unexpected_keys))
if incompatible.missing_keys:
    print("first missing:", incompatible.missing_keys[:20])
if incompatible.unexpected_keys:
    print("first unexpected:", incompatible.unexpected_keys[:20])

# 5. На устройство
model.to(device)
model.eval()

# 6. Подменяем trainer
trainer.model = model

def preview_predictions(num_examples: int = 1):
    subset = dataset['val'].select(range(min(num_examples, len(dataset['val']))))
    outputs = trainer.predict(subset)

    preds = outputs.predictions
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)

    decoded = processor.batch_decode(preds, skip_special_tokens=False)

    for idx, sample in enumerate(decoded):
        print('=' * 80)
        print(f'Пример {idx}')
        print(sample[:4000])
        print('\nGT:')
        print(subset[idx]['transcription'][:4000])
        
# 7. Смотрим predictions
preview_predictions(5)


saved tokenizer size: 4107


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

missing keys: 0
unexpected keys: 0
checkpoint missing keys: 509
checkpoint unexpected keys: 0
first missing: ['model.vision_model.class_embedding', 'model.vision_model.patch_embedding.weight', 'model.vision_model.gated_positional_embedding.gate', 'model.vision_model.gated_positional_embedding.embedding', 'model.vision_model.gated_positional_embedding.tile_embedding.weight', 'model.vision_model.pre_tile_positional_embedding.gate', 'model.vision_model.pre_tile_positional_embedding.embedding.weight', 'model.vision_model.post_tile_positional_embedding.gate', 'model.vision_model.post_tile_positional_embedding.embedding.weight', 'model.vision_model.layernorm_pre.weight', 'model.vision_model.layernorm_pre.bias', 'model.vision_model.layernorm_post.weight', 'model.vision_model.layernorm_post.bias', 'model.vision_model.transformer.layers.0.self_attn.q_proj.weight', 'model.vision_model.transformer.layers.0.self_attn.k_proj.weight', 'model.vision_model.transformer.layers.0.self_attn.v_proj.weight'

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


computing LER...: 100%|██████████████████████████████████████████████████████████████████| 5/5 [00:02<00:00,  2.06it/s]


Пример 0
<|begin_of_abc|>X:1
T:Title
%%score
L:1/8
Q:1/4=108
M:4/4
I:linebreak $
K:C
V:1 treble
V:1
 [GA]2 A2 B2 c2 B4 A2 A2 A2 B2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2 G2 B2 c2 B2 A2 G4 z4 [GA]2 A2 B2 c2 B4 A2 A2 G2 G2 z2